MD# ADVANCE NATURAL LANGUAGE PROCESSING (AML23702) - Lab Exercise 2

**Name:** P Prithvi Raj Kandan &nbsp;|&nbsp; **USN:** 1GA23AI033 &nbsp;|&nbsp; Dept. of AI & ML, Global Academy of Technology

**Aim:** Accept a question and retrieve the most relevant sentence from a small document collection (TF-IDF + cosine similarity).

In [1]:
%pip install -q scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = Path("../data")
documents = [ln.strip() for ln in (DATA / "glossary.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
glossary_questions = [ln.strip() for ln in (DATA / "glossary_questions.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]

# Split documents into sentences (each example is already one sentence).
sentences = documents

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(sentences)

def answer_question(question):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, document_vectors).flatten()
    best_index = scores.argmax()

    print("Question:", question)
    print("Retrieved answer:", sentences[best_index])
    print("Similarity score:", round(float(scores[best_index]), 3))

answer_question(glossary_questions[0])


Question: What is WordNet used for?
Retrieved answer: WordNet is a lexical database containing words, synonyms, and definitions.
Similarity score: 0.267


In [3]:
# Try another question from data/glossary_questions.txt
question = glossary_questions[1]
answer_question(question)


Question: Which library performs named entity recognition?
Retrieved answer: SpaCy is a Python library used for natural language processing and named entity recognition.
Similarity score: 0.666


MD### Same method on a multi-sentence document collection (used by the web frontend)
Each document is split into sentences with NLTK, so the answer is a single sentence, not a whole document.

In [4]:
import nltk
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

docs = [ln.strip() for ln in (DATA / "documents.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
questions = [ln.strip() for ln in (DATA / "question.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
questions += [ln.strip() for ln in (DATA / "suggestions.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
sentences = [s for d in docs for s in nltk.sent_tokenize(d)]
vec = TfidfVectorizer(stop_words="english")
sent_vectors = vec.fit_transform(sentences)
print(len(docs), "documents ->", len(sentences), "sentences")
print("Loaded", (DATA / "documents.txt").resolve())

def ask(question, top_k=3):
    scores = cosine_similarity(vec.transform([question]), sent_vectors).flatten()
    print("Q:", question)
    for rank, i in enumerate(scores.argsort()[::-1][:top_k], start=1):
        print(f"  {rank}. ({scores[i]:.3f}) {sentences[i]}")
    print()

for question in questions:
    ask(question)


5 documents -> 9 sentences
Loaded /workspace/AAT2/data/documents.txt
Q: When is Engineers' Day celebrated in India?
  1. (0.756) Engineers' Day is celebrated in India on 15 September in his honour.
  2. (0.000) It is lit with thousands of bulbs during Dasara.
  3. (0.000) The Mysore Palace was rebuilt after a fire and completed in 1912.

Q: Where is ISRO headquartered?
  1. (0.418) The Indian Space Research Organisation is headquartered in Bengaluru.
  2. (0.000) It is lit with thousands of bulbs during Dasara.
  3. (0.000) The Mysore Palace was rebuilt after a fire and completed in 1912.

Q: Which building houses the Karnataka legislature?
  1. (0.665) The Vidhana Soudha in Bengaluru houses the state legislature of Karnataka.
  2. (0.225) Kannada is the official language of Karnataka.
  3. (0.000) It is lit with thousands of bulbs during Dasara.

Q: When was the Mysore Palace completed?
  1. (0.775) The Mysore Palace was rebuilt after a fire and completed in 1912.
  2. (0.000) It is l